### RAG evaluation

In [ ]:
from langsmith import traceable
from retrieval.hybridSearch import HybridSearch
# from generation.llm_client import LLMClient
from langchain.chat_models import init_chat_model
hybrid_search = HybridSearch()
# llm_client  = LLMClient()
llm = init_chat_model("openai/gpt-oss-safeguard-20b")
@traceable()
def rag_bot(question:str)->dict:
    ## relevant context 
    docs = hybrid_search.hybrid_search_with_rrf(question)
    doc_string = "".join(m.text for m in docs.metadata)

    instruction = f""" you are a helpful assistant who is good at analyzing source information and answering questions. use the following dource document to answer the user's question, if u don't know the answer just say that you don't know, use thre sentences maiximum and the keep the answer concise.

    documents:
    {doc_string}
    """
    ## llm invoke
    ai_answer = llm.invoke([
        {"role":"user", "content":question},
        {"role":"system", "content":instruction}
    ])

    return {"answer":ai_answer.content, "documenst":docs}


In [ ]:
rag_bot("what is bla bla ??")

### DataSet

In [ ]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs":{"question":""},
        "outputs":{"answer":""}
    },
    {
        "inputs":{"question":""},
        "outputs":{"answer":""}
    },
    {
        "inputs":{"question":""},
        "outputs":{"answer":""}
    },
    {
        "inputs":{"question":""},
        "outputs":{"answer":""}
    },
]
## create the dataset and examples in langsmith
dataset_name = "contract assistant test" 
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)


### Evaluators

1- correctness: Response vs reference answer:

> goal: Measure "how similar/correct i sthe RAG chain answer, relative to a ground-truth answer"

> Evaluator: Use LLM-as-judge to access answer correctness